In [46]:
import pandas as pd
import glob
import numpy as np
import os

In [47]:
def clean_and_merge_pihps(file_pattern, target_commodity="Cabai Rawit Merah"):
    """
    Fungsi untuk menjahit file PIHPS tahunan, memfilter komoditas, 
    dan merombaknya menjadi Time-Series murni.
    """
    print(f"Mencari file dengan pola: {file_pattern}")
    all_files = glob.glob(file_pattern)
    
    if not all_files:
        raise ValueError("File tidak ditemukan, Cek kembali path dan pola nama file.")
        
    df_list = []
    
    for file in all_files:
        print(f"   Memproses: {file}")
        try:
            # PIHPS seringkali memiliki header yang berantakan, kita baca semua sebagai string
            df_temp = pd.read_csv(file, dtype=str)
        except Exception:
            try:
                df_temp = pd.read_excel(file, dtype=str)
            except Exception as e:
                print(f"Gagal membaca {file}: {e}")
                continue
            
        # 1. Cari kolom yang mengandung kata "Komoditas"
        col_komoditas = [col for col in df_temp.columns if 'Komoditas' in str(col)]
        if not col_komoditas:
            continue
            
        komoditas_name = col_komoditas[0]
        
        # 2. Filter HANYA Cabai Rawit Merah (case-insensitive)
        # Hati-hati dengan whitespace atau karakter aneh dari Excel
        df_filtered = df_temp[df_temp[komoditas_name].astype(str).str.contains(target_commodity, case=False, na=False)].copy()
        
        if df_filtered.empty:
            print(f"Peringatan: Tidak ditemukan '{target_commodity}' di {file}")
            continue
            
        # 3. Hapus kolom "No" atau "ID" jika ada, sisakan hanya kolom tanggal dan komoditas
        cols_to_keep = [col for col in df_filtered.columns if 'No' not in str(col) and col != 'ID']
        df_filtered = df_filtered[cols_to_keep]
        
        # 4. Melt (Unpivot) dari Wide ke Long
        df_melted = df_filtered.melt(id_vars=[komoditas_name], var_name='Tanggal', value_name='Harga_Cabai')
        
        # Buang nama komoditas karena tidak perlu lagi
        df_melted = df_melted.drop(columns=[komoditas_name])
        
        df_list.append(df_melted)

    if not df_list:
        raise ValueError(f"Gagal memproses data. Pastikan kata '{target_commodity}' benar-benar ada di dalam file.")

    # 5. Gabungkan semua tahun menjadi satu DataFrame
    df_master = pd.concat(df_list, ignore_index=True)
    
    # 6. Cleaning Waktu (Parsing Datetime)
    # Parameter dayfirst=True sangat PENTING untuk format tanggal Indonesia (DD/MM/YYYY)
    df_master['Tanggal'] = pd.to_datetime(df_master['Tanggal'].astype(str).str.strip(), dayfirst=True, errors='coerce')
    
    # Hapus baris yang gagal di-parse menjadi tanggal (misalnya karena itu nama komoditas atau string kosong)
    df_master = df_master.dropna(subset=['Tanggal']) 
    
    # 7. Cleaning Harga
    df_master['Harga_Cabai'] = df_master['Harga_Cabai'].replace('-', np.nan)
    # Hapus titik ribuan, spasi, koma (jika ada), dan ubah ke float
    df_master['Harga_Cabai'] = df_master['Harga_Cabai'].astype(str).str.replace(r'[^\d.]', '', regex=True)
    df_master['Harga_Cabai'] = pd.to_numeric(df_master['Harga_Cabai'], errors='coerce')
    
    # 8. Set Index, Urutkan Waktu, dan Tangani Missing Values (Hari Libur)
    df_master = df_master.sort_values('Tanggal').set_index('Tanggal')
    
    # Menghapus duplikasi tanggal (jika ada) dan simpan yang pertama kali muncul
    df_master = df_master[~df_master.index.duplicated(keep='first')]
    
    # Mengisi hari-hari yang kosong menggunakan reindexing agar kalender menjadi lengkap (kontinu)
    min_date = df_master.index.min()
    max_date = df_master.index.max()
    full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
    
    # Menyisipkan hari libur yang hilang dan mengisi harganya dengan Forward Fill
    df_master = df_master.reindex(full_date_range)
    df_master['Harga_Cabai'] = df_master['Harga_Cabai'].ffill()
    
    df_master.index.name = 'Tanggal'
    
    return df_master

In [48]:
# ==========================================
# EKSEKUSI UNTUK GARUT & BANDUNG (DIPERBAIKI)
# ==========================================
path_garut = "../data/01_raw/pihps/PIHPS_Garut_Produsen*.xlsx"
path_bandung = "../data/01_raw/pihps/PIHPS_Bandung_Grosir*.xlsx"

print("--- Menyiapkan Data Garut ---")
files_garut = glob.glob(path_garut)
if not files_garut:
    raise FileNotFoundError(
        f"Tidak ditemukan file dengan pola: {path_garut}\n"
        "Pastikan notebook dijalankan dari root project dan nama file cocok."
    )

# 1. Proses Data Garut dan Rename Kolom
df_harga_garut = clean_and_merge_pihps(path_garut)
df_harga_garut = df_harga_garut.rename(columns={'Harga_Cabai': 'Harga_Garut'})
print("\nGarut Selesai. Cuplikan Data:")
display(df_harga_garut.head())

print("\n--- Menyiapkan Data Bandung ---")
# 2. Proses Data Bandung dan Rename Kolom
df_harga_bandung = clean_and_merge_pihps(path_bandung)
df_harga_bandung = df_harga_bandung.rename(columns={'Harga_Cabai': 'Harga_Bandung'})
print("\nBandung Selesai. Cuplikan Data:")
display(df_harga_bandung.head())

--- Menyiapkan Data Garut ---
Mencari file dengan pola: ../data/01_raw/pihps/PIHPS_Garut_Produsen*.xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Garut_Produsen (2022).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Garut_Produsen (2023).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Garut_Produsen (2024).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Garut_Produsen (2025).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Garut_Produsen (Jan 1 - 1 Apr 2026).xlsx

Garut Selesai. Cuplikan Data:


,Harga_Garut
Tanggal,
2022-01-03,65000.0
2022-01-04,65000.0
2022-01-05,65000.0
2022-01-06,65000.0
2022-01-07,65000.0



--- Menyiapkan Data Bandung ---
Mencari file dengan pola: ../data/01_raw/pihps/PIHPS_Bandung_Grosir*.xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Bandung_Grosir (1 Jan - 1 Apr 2026).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Bandung_Grosir (2022).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Bandung_Grosir (2023).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Bandung_Grosir (2024).xlsx
   Memproses: ../data/01_raw/pihps\PIHPS_Bandung_Grosir (2025).xlsx

Bandung Selesai. Cuplikan Data:


,Harga_Bandung
Tanggal,
2022-01-03,77400.0
2022-01-04,77050.0
2022-01-05,75450.0
2022-01-06,64250.0
2022-01-07,64250.0


In [49]:
# ==========================================
# EKSPOR KE INTERIM DATA
# ==========================================
import os

# 1. Pastikan folder tujuan ada
output_dir = "../data/02_interim"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"\nMembuat folder baru: {output_dir}")

# 2. Simpan Data Harga Garut (Produsen)
file_garut = os.path.join(output_dir, "PIHPS_Garut_Cleaned.csv")
df_harga_garut.to_csv(file_garut, index=True)
print(f"Tersimpan: {file_garut}")

# 3. Simpan Data Harga Bandung (Grosir)
file_bandung = os.path.join(output_dir, "PIHPS_Bandung_Cleaned.csv")
df_harga_bandung.to_csv(file_bandung, index=True)
print(f"Tersimpan: {file_bandung}")

# 4. Verifikasi Akhir
print(f"\nProses Merging & Cleaning Selesai.")
print(f"Total baris data Garut: {len(df_harga_garut)}")
print(f"Total baris data Bandung: {len(df_harga_bandung)}")

Tersimpan: ../data/02_interim\PIHPS_Garut_Cleaned.csv
Tersimpan: ../data/02_interim\PIHPS_Bandung_Cleaned.csv

Proses Merging & Cleaning Selesai.
Total baris data Garut: 1550
Total baris data Bandung: 1550
